<a href="https://colab.research.google.com/github/smkim0508/cos484-notes/blob/main/A4_Programming_Cross_Entropy_Loss_on_GPU_(COS484_S2026).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook for Programming in Problem 4
Welcome to the programming portion of the assignment! Each assignment throughout the semester will have a written portion and a programming portion. We will be using [Google Colab](https://colab.research.google.com/notebooks/intro.ipynb#recent=true), so if you have never used it before, take a quick look through this introduction: [Working with Google Colab](https://docs.google.com/document/d/1LlnXoOblXwW3YX-0yG_5seTXJsb3kRdMMRYqs8Qqum4/edit?usp=sharing).

## Cross Entropy Loss on GPU
In this problem you will benchmark the cross-entropy loss in PyTorch and measure its efficiency relative to the theoretical memory bandwidth of the GPU. You will predict performance before measuring, then explain any discrepancies — this is the core skill of systems performance analysis.
Setup: Use the following parameters throughout:

-  Batch size: B = 4,096
-  Vocabulary size: V = 128,256
-  Logits dtype: torch.bfloat16
-  Reduction: ’none’
- Device: GPU (use a Colab GPU runtime, e.g., T4 or A100)
-  Report your PyTorch version (e.g., torch. version ) and GPU model in your notebook. Results can vary significantly across versions.

## Generate random input logits and targets

In [12]:
# Generate random input logits and targets:
import torch
B, V = 4096, 128256
# NOTE: I had to change the datatype to float16, not bfloat16 to be compatible with T4
logits = torch.randn(B, V, dtype=torch.float16 , device='cuda',
requires_grad=True)
targets = torch.randint(0, V, (B,), device='cuda')

# Questions

**(a) Predict before you measure. (8 points)**

Before writing any benchmarking code, use your answers from Theory Problem 1 to predict the performance of an ideal (perfectly memory-bound) cross-entropy kernel on your GPU.

**preliminary: I am using T4 on this colab session.

(i) Look up the peak memory bandwidth of your Colab GPU. For this assignment, if you are using an A100, assume the A100 SXM bandwidth of ∼2039 GB/s; if you are using a T4, use ∼300 GB/s. Using the total bytes from Theory Problem 1(c), compute the minimum possible time (in ms) for the forward pass and backward pass, assuming the kernel runs at 100% of peak memory bandwidth. (4 points)

For T4 with around 300 GB/s bandwidth, forward pass with B=4096, V=128256 is 2BV + 12B ~= 1.05GB, and 1.05GB/(300GB/s) = 3.5ms.

For backward, 4BV + 12B ~= 2.1GB, 2.1GB/(300GB/s) = 7ms.

So in total, minimum possible time is ~= 10ms for both forward and backward pass combined.

(ii) Based on your arithmetic intensity analysis from Theory Problem 1(c), explain why memory bandwidth (not TFLOPS) is the right performance limiter and the right metric to evaluate the efficiency of cross- entropy. (4 points)

Using A100 as reference (since this is what I solved problem 1c for), we have AI values of 1.5 and 1.25 whereas the compute-to-bandwidth ratio is somewhere around 150 FLOPs/byte. The AI values are far lower than the compute-to-bandwidth ratio, which indicates that memory bandwidth is the limiting factor for performance limiter. The T4 has a similar compute-to-bandwidth ratio (~217 FLOPs/byte from NVIDIA's spec sheet), so the same
conclusion holds that the AI values are far below the threshold and memory is the key performance bottleneck.

In [3]:
# check version for colab
print(torch.cuda.get_device_name())
print(torch.__version__)

Tesla T4
2.10.0+cu128



**(b) Benchmark PyTorch eager mode. (12 points)**


(i) Write a benchmarking function that measures the wall-clock time (in milliseconds) for the cross-entropy forward pass and backward pass separately. Use torch.cuda.Event with enable timing=True for accurate GPU timing. After recording the end event, make sure to synchronize the GPU (e.g., with torch.cuda.synchronize() or end event.synchronize()) before reading the elapsed time. Run each measurement multiple times (e.g., 100 iterations) and report the median time. Make sure to do a few warmup iterations before timing. (5 points)

Forward median: 16.654 ms

Backward median: 21.375 ms

(ii) Compute the achieved memory bandwidth (in GB/s) for the forward and backward passes: Achieved BW (GB/s) = Total bytes (GB) / Time (s)
    
How does the measured time compare to your prediction from part (a)? If there is a significant gap, explain why PyTorch eager mode is slower than the ideal. What extra memory traffic or overhead might the eager implementation incur? (7 points)

Hint: Think about whether the eager implementation fuses all the operations into a single kernel, or whether it materializes intermediate tensors (e.g., the full softmax output) to HBM.

Forward achieved bandwidth: 63.1 GB/s (21.0% of peak)

Backward achieved bandwidth: 98.3 GB/s (32.8% of peak)

As shown by the results, assuming 300 GB/s is the peak performance from T4, we are very far from achieving this bandwidth.

This issue exists because PyTorch eager mode does not fuse all operations into a single kernel. Instead, each operation (subtract
max, exp, sum, divide, log, etc.) runs as a separate CUDA kernel, and each kernel reads its input from memory (HBM) and writes its output back to memory. As such, the large intermediate tensors, which are costly to move around, including BxV softmax output, is constantly transferred to and from memory across operations, since they do not share the same kernel (same SRAM). The theoretical minimum time assumes every operation exists in the same kernel and therefore does not need intermediate transfer times and only outputs the final vector.


Code for part (b):

In [13]:
# benchmarking function
import torch
import torch.nn.functional as F

def benchmark_cross_entropy(logits, targets, n_warmup=10, n_iter=100):
    """
    Benchmark forward and backward pass of cross-entropy loss separately.
    Uses torch.cuda.Event for accurate GPU timing.
    Returns median forward and backward times in milliseconds.
    """
    fwd_times = []
    bwd_times = []

    # On each benchmark run, do warmup iterations to lazy init gpu
    for _ in range(n_warmup):
        loss = F.cross_entropy(logits, targets, reduction='none')
        grad_output = torch.ones_like(loss)
        loss.backward(grad_output)
        logits.grad = None

    # Timed iterations following warm-ups
    for _ in range(n_iter):
        # Forward timing
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        loss = F.cross_entropy(logits, targets, reduction='none')
        end.record()
        end.synchronize()
        fwd_times.append(start.elapsed_time(end))

        # Backward timing
        grad_output = torch.ones_like(loss)
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        loss.backward(grad_output)
        end.record()
        end.synchronize()
        bwd_times.append(start.elapsed_time(end))

        logits.grad = None

    fwd_median = sorted(fwd_times)[n_iter // 2]
    bwd_median = sorted(bwd_times)[n_iter // 2]

    print(f"Forward median: {fwd_median:.3f} ms")
    print(f"Backward median: {bwd_median:.3f} ms")
    return fwd_median, bwd_median

fwd_ms, bwd_ms = benchmark_cross_entropy(logits, targets)

Forward median: 16.654 ms
Backward median: 21.375 ms


In [14]:
# simple test to find the acheived bandwidth

B, V = 4096, 128256

fwd_bytes = 2 * B * V + 12 * B
bwd_bytes = 4 * B * V + 12 * B

fwd_bandwidth = (fwd_bytes / 1e9) / (fwd_ms / 1000)
bwd_bandwidth = (bwd_bytes / 1e9) / (bwd_ms / 1000)

peak_bw = 300 # T4 peak bandwidth in GB/s

print(f"Forward achieved bandwidth: {fwd_bandwidth:.1f} GB/s ({fwd_bandwidth / peak_bw * 100:.1f}% of peak)")
print(f"Backward achieved bandwidth: {bwd_bandwidth:.1f} GB/s ({bwd_bandwidth / peak_bw * 100:.1f}% of peak)")

Forward achieved bandwidth: 63.1 GB/s (21.0% of peak)
Backward achieved bandwidth: 98.3 GB/s (32.8% of peak)


**(c) Benchmark torch.compile mode. (10 points)**

(i) Wrap the cross-entropy computation with torch.compile and repeat the benchmarking from part (b). Note that the first call triggers compilation — make sure your warmup accounts for this. Report the achieved memory bandwidth. (4 points)

Benchmarked results:

Forward  median: 5.838 ms

Backward median: 9.257 ms

Forward achieved bandwidth: 180.0 GB/s (60.0% of peak)

Backward achieved bandwidth: 227.0 GB/s (75.7% of peak)

(ii) How does torch.compile compare to eager mode and to your ideal prediction? Explain what torch.compile is likely doing differently (e.g., kernel fusion) that accounts for the performance difference. (6 points)

Compiled forward achieves 180.0 GB/s (60.0% of peak), compared to eager mode's 63.1 GB/s (21.0%). Compiled backward achieves 227.0 GB/s (75.7%), compared to eager's 98.3 GB/s (32.8%). This is roughly a 3 fold improvement,
though still below the ideal prediction of 300 GB/s.

In this case, torch.compile uses kernel fusion to achieve this speedup. Instead of
running each operation as a separate
kernel that unnecessarily writes intermediate states to the slow main GPU memory (HBM), as previously observed in part (b), torch.compile here fuses multiple operations into a single Triton kernel. As a result, the BxV logits are loaded from main memory once, then the intermediate processing takes place in shared SRAM, and only the final output is written back to main memory, drastically improving performance compared to previous benchmark.

However, this still does not reach ideal prediction, which is likely due to overhead such as kernel launch latency, memory access not always being ideal in practice (possible misses, failures), and possibly the fused kernel not utilizing 100% of its theoretical memory bandwidth.

Code for part (c):

In [16]:
def cross_entropy_fn(logits, targets):
    """
    Wrapper for cross-entropy forward pass to use with torch.compile.
    """
    return F.cross_entropy(logits, targets, reduction='none')

compiled_cross_entropy = torch.compile(cross_entropy_fn)

def benchmark_compiled(logits, targets, n_warmup=10, n_iter=100):
    """
    Benchmark compiled cross-entropy forward and backward pass.
    Extra warmup to account for compilation on first call.
    """
    fwd_times = []
    bwd_times = []

    # On each benchmark run, do warmup iterations to lazy init gpu
    for _ in range(n_warmup):
        loss = compiled_cross_entropy(logits, targets)
        grad_output = torch.ones_like(loss)
        loss.backward(grad_output)
        logits.grad = None

    # Timed iterations following warm-ups
    for _ in range(n_iter):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        loss = compiled_cross_entropy(logits, targets)
        end.record()
        end.synchronize()
        fwd_times.append(start.elapsed_time(end))

        grad_output = torch.ones_like(loss)
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)
        start.record()
        loss.backward(grad_output)
        end.record()
        end.synchronize()
        bwd_times.append(start.elapsed_time(end))

        logits.grad = None

    fwd_median = sorted(fwd_times)[n_iter // 2]
    bwd_median = sorted(bwd_times)[n_iter // 2]

    print(f"Forward  median: {fwd_median:.3f} ms")
    print(f"Backward median: {bwd_median:.3f} ms")
    return fwd_median, bwd_median

compiled_fwd_ms, compiled_bwd_ms = benchmark_compiled(logits, targets)

# Compute achieved bandwidth
peak_bw = 300
fwd_bytes = 2 * B * V + 12 * B
bwd_bytes = 4 * B * V + 12 * B

compiled_fwd_bandwidth = (fwd_bytes / 1e9) / (compiled_fwd_ms / 1000)
compiled_bwd_bandwidth = (bwd_bytes / 1e9) / (compiled_bwd_ms / 1000)

print(f"Forward achieved bandwidth: {compiled_fwd_bandwidth:.1f} GB/s ({compiled_fwd_bandwidth / peak_bw * 100:.1f}% of peak)")
print(f"Backward achieved bandwidth: {compiled_bwd_bandwidth:.1f} GB/s ({compiled_bwd_bandwidth / peak_bw * 100:.1f}% of peak)")

Forward  median: 5.838 ms
Backward median: 9.257 ms
Forward achieved bandwidth: 180.0 GB/s (60.0% of peak)
Backward achieved bandwidth: 227.0 GB/s (75.7% of peak)



**(d) Inspecting the compiled kernel. (10 points)**

When torch.compile compiles your function, it generates Triton kernel code under the hood.

(i) Use the environment variable TORCH LOGS=output code or torch. dynamo.config.log level to dump the generated Triton kernel code for the cross-entropy forward pass. Include the generated code (or relevant excerpts) in your notebook. (3 points)


(ii) Annotate the generated kernel: identify which lines correspond to (1) computing the max for numerical stability, (2) the exp and sum (softmax denominator), (3) the log-sum-exp, and (4) the final loss computation. Relate these back to the formulas you derived in Theory Problem 1(a). (7 points)

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)



Code for part (d):

In [22]:
# Dumps the raw kernel code generated

import os
import torch
import torch.nn.functional as F

# Clear environment variable and reset
os.environ.pop("TORCH_LOGS", None)
os.environ.pop("TORCH_LOGS_OUT", None)
torch._dynamo.reset()

# Use programmatic logging
torch._logging.set_logs(output_code=True)

def cross_entropy_fn(logits, targets):
    """
    Wrapper for cross-entropy forward pass.
    """
    return F.cross_entropy(logits, targets, reduction='none')

compiled_cross_entropy = torch.compile(cross_entropy_fn)

loss = compiled_cross_entropy(logits, targets)

V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] Output code: 
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] # AOT ID: ['1_forward']
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] from ctypes import c_void_p, c_long, c_int
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] import torch
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] import math
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] import random
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] import os
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] import tempfile
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__output_code] from math import inf, nan
V0415 03:21:40.007000 1679 torch/_inductor/codecache.py:1250] [0/0] [__o

In [ ]:
# Raw part of the kernel code that's relevant

_tmp3_max = tl.full([XBLOCK, R0_BLOCK], float('-inf'), tl.float32)
_tmp3_sum = tl.zeros([XBLOCK, R0_BLOCK], tl.float32)
for r0_offset in range(0, r0_numel, R0_BLOCK):
    r0_index = r0_offset + r0_base
    r0_mask = r0_index < r0_numel
    roffset = r0_offset
    rindex = r0_index
    r0_1 = r0_index
    tmp0 = tl.load(in_ptr0 + (r0_1 + 128256*x0), r0_mask, eviction_policy='evict_last', other=0.0).to(tl.float32)
    tmp1 = tmp0.to(tl.float32)
    tmp2 = tl.broadcast_to(tmp1, [XBLOCK, R0_BLOCK])

    _tmp3_max_next, _tmp3_sum_next = triton_helpers.online_softmax_combine(
        _tmp3_max, _tmp3_sum, tmp2, False
    )

    _tmp3_max = tl.where(r0_mask, _tmp3_max_next, _tmp3_max)
    _tmp3_sum = tl.where(r0_mask, _tmp3_sum_next, _tmp3_sum)

tmp3, tmp4 = triton_helpers.online_softmax_reduce(
    _tmp3_max, _tmp3_sum, 1, False)
tmp3 = tmp3[:, None]
tmp4 = tmp4[:, None]
tl.store(out_ptr0 + (x0), tmp3, None)
tmp6 = tl.load(in_ptr1 + (x0), None, eviction_policy='evict_last')
tmp5 = tl_math.log(tmp4)
tmp7 = tl.full([1, 1], -100, tl.int64)
tmp8 = tmp6 != tmp7
tmp9 = tl.full([1, 1], 0, tl.int64)
tmp10 = tl.where(tmp8, tmp6, tmp9)
tmp11 = tl.full([1, 1], 128256, tl.int32)
tmp12 = tmp10 + tmp11
tmp13 = tmp10 < 0
tmp14 = tl.where(tmp13, tmp12, tmp10)
tl.device_assert((0 <= tmp14) & (tmp14 < 128256), "index out of bounds: 0 <= tmp14 < 128256")
tmp16 = tl.load(in_ptr0 + (tmp14 + 128256*x0), None, eviction_policy='evict_last').to(tl.float32)
tmp17 = tmp16.to(tl.float32)
tmp18 = tmp17 - tmp3
tmp19 = tmp18 - tmp5
tmp20 = tmp19.to(tl.float32)
tmp21 = -tmp20
tmp22 = 0.0
tmp23 = tl.where(tmp8, tmp21, tmp22)
tl.debug_barrier()
tl.store(in_out_ptr0 + (x0), tmp5, None)
tl.store(out_ptr1 + (x0), tmp23, None)



(ii) Annotate the generated kernel: identify which lines correspond to (1) computing the max for numerical stability, (2) the exp and sum (softmax denominator), (3) the log-sum-exp, and (4) the final loss computation. Relate these back to the formulas you derived in Theory Problem 1(a). (7 points)


In [ ]:
# Annotated version of the kernel code

# (1) Computing max and (2) exp + sum (softmax denominator)
# The kernel code seems to utilize the online softmax algorithm, computing m = max(x_j) and sum(exp(x_j - m)) in a single pass, re-scaling the sum whenever a new max is found.
# - This avoids reading logits twice.
_tmp3_max = tl.full([XBLOCK, R0_BLOCK], float('-inf'), tl.float32)
_tmp3_sum = tl.zeros([XBLOCK, R0_BLOCK], tl.float32)
for r0_offset in range(0, r0_numel, R0_BLOCK):
    r0_index = r0_offset + r0_base
    r0_mask = r0_index < r0_numel
    roffset = r0_offset
    rindex = r0_index
    r0_1 = r0_index
    tmp0 = tl.load(in_ptr0 + (r0_1 + 128256*x0), r0_mask, eviction_policy='evict_last', other=0.0).to(tl.float32) # Load logit x_j
    tmp1 = tmp0.to(tl.float32) # Cast to fp32
    tmp2 = tl.broadcast_to(tmp1, [XBLOCK, R0_BLOCK])

    # Updates running max and running sum(exp(x_j - current_max))
    _tmp3_max_next, _tmp3_sum_next = triton_helpers.online_softmax_combine(
        _tmp3_max, _tmp3_sum, tmp2, False
    )

    _tmp3_max = tl.where(r0_mask, _tmp3_max_next, _tmp3_max)
    _tmp3_sum = tl.where(r0_mask, _tmp3_sum_next, _tmp3_sum)

# Finalize: tmp3 = m (max), tmp4 = sum(exp(x_j - m))
tmp3, tmp4 = triton_helpers.online_softmax_reduce(
    _tmp3_max, _tmp3_sum, 1, False)
tmp3 = tmp3[:, None] # m = max_j(x_j)
tmp4 = tmp4[:, None] # sum_j(exp(x_j - m))
tl.store(out_ptr0 + (x0), tmp3, None) # Store max (saved for backward)

# (3) Log-sum-exp: finding log(sum(exp(x_j - m)))
tmp5 = tl_math.log(tmp4)

# (4) Final loss computation
# Handle ignore_index = -100 for padding tokens
tmp6 = tl.load(in_ptr1 + (x0), None, eviction_policy='evict_last') # Load target y
tmp7 = tl.full([1, 1], -100, tl.int64)
tmp8 = tmp6 != tmp7 # Check if target is not ignore_index
tmp9 = tl.full([1, 1], 0, tl.int64)
tmp10 = tl.where(tmp8, tmp6, tmp9) # Replace ignore_index with 0
tmp11 = tl.full([1, 1], 128256, tl.int32)
tmp12 = tmp10 + tmp11
tmp13 = tmp10 < 0
tmp14 = tl.where(tmp13, tmp12, tmp10) # Handle negative indices
tl.device_assert((0 <= tmp14) & (tmp14 < 128256), "index out of bounds: 0 <= tmp14 < 128256")
tmp16 = tl.load(in_ptr0 + (tmp14 + 128256*x0), None, eviction_policy='evict_last').to(tl.float32) # Load x_y
tmp17 = tmp16.to(tl.float32)
tmp18 = tmp17 - tmp3 # x_y - m
tmp19 = tmp18 - tmp5 # x_y - m - log(sum(exp(x_j - m))) = log_softmax(x_y)
tmp20 = tmp19.to(tl.float32)
tmp21 = -tmp20 # Negate: loss = -(x_y - m - log(sum(exp(x_j - m)))), which equals: -x_y + m + log(sum(exp(x_j - m)))
tmp22 = 0.0
tmp23 = tl.where(tmp8, tmp21, tmp22) # Zero out loss for ignore_index tokens
tl.debug_barrier()
tl.store(in_out_ptr0 + (x0), tmp5, None) # Store log(sum(exp)) for backward
tl.store(out_ptr1 + (x0), tmp23, None) # Store final loss

# Problem 2: Bonus — Cross-Entropy Speed Competition

See task specification and submission instruction in the problem set PDF. Submission window is open from April 7 to April 16. If you're submitting for bonus credits, please include your optimization journal here documenting your optimization process:


- What approaches did you try? (e.g., torch.compile tricks, custom Triton kernels, fused forward+backward,
memory layout changes, block size tuning, etc.)
- For each approach, report the timing and explain why it was faster or slower than your previous best.
- What is the achieved memory bandwidth of your final submission, and what fraction of peak A100 bandwidth (2039 GB/s) does it reach? What do you think is preventing you from reaching 100%?


If you prefer to document your optimization in a separate PDF, please also make a note here.


# LLM Prompts

If you used an AI tool to complete any part of this assignment, please paste all prompts you used to produce your final code/responses in the box below and answer the following reflection question.

Prompts Used:
*   
*   



**Reflection: What parts of the AI generated output required modification or improvement? Describe the feedback you gave the tool to produce your final output or any changes you had to make on your own.**

TODO: ANSWER THE QUESTION HERE (DOUBLE-CLICK TO EDIT)